In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical 
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional

In [ ]:
SONNETS_FILE = './sonnets.txt'
with open('./sonnets.txt') as f:
    data = f.read()

corpus = data.lower().split("\n")
for i in range(5):
  print(corpus[i])

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1
print(total_words)

In [ ]:
corpus[0]

In [ ]:
tokenizer.texts_to_sequences(corpus[0])

In [ ]:
tokenizer.texts_to_sequences([corpus[0]])

In [ ]:
tokenizer.texts_to_sequences([corpus[0]])[0]

In [ ]:
def n_gram_seqs(corpus, tokenizer):
    input_sequences = []
    for line in corpus:
      token_list = tokenizer.texts_to_sequences([line])[0]
      for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)
    return input_sequences

In [ ]:
first_example_sequence = n_gram_seqs([corpus[0]], tokenizer)
first_example_sequence

In [ ]:
next_3_examples_sequence = n_gram_seqs(corpus[1:4], tokenizer)
next_3_examples_sequence

In [ ]:
input_sequences = n_gram_seqs(corpus, tokenizer)
max_sequence_len = max([len(x) for x in input_sequences])

In [ ]:
def pad_seqs(input_sequences, maxlen):
    padded_sequences = pad_sequences(input_sequences, maxlen=maxlen, padding='pre')
    return padded_sequences

In [ ]:
first_padded_seq = pad_seqs(first_example_sequence, len(first_example_sequence))
first_padded_seq

In [ ]:
next_3_padded_seq = pad_seqs(next_3_examples_sequence, max([len(s) for s in next_3_examples_sequence]))
next_3_padded_seq

In [ ]:
input_sequences = pad_seqs(input_sequences, max_sequence_len)

In [ ]:
def features_and_labels(input_sequences, total_words):
    features = input_sequences[:,:-1]
    labels = input_sequences[:,-1]
    one_hot_labels = to_categorical(labels, num_classes=total_words)
    return features, one_hot_labels

In [ ]:
first_features, first_labels = features_and_labels(first_padded_seq, total_words)
first_features

In [ ]:
features, labels = features_and_labels(input_sequences, total_words)

In [ ]:
import tensorflow as tf
def create_model(total_words, max_sequence_len):
    model = Sequential()
    model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))

    model.add(Bidirectional(tf.keras.layers.GRU(32))),
    model.add(Dense(512, activation='relu'))
    model.add(Dense(total_words, activation='softmax'))

    model.compile(loss='categorical_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  metrics=['accuracy'])
    return model

In [ ]:
model = create_model(total_words, max_sequence_len)
history = model.fit(features, labels, epochs=50, verbose=1, batch_size=64)

In [ ]:
acc = history.history['accuracy']
loss = history.history['loss']

epochs = range(len(acc))

plt.plot(epochs, acc, 'b', label='Training accuracy')
plt.title('Training accuracy')

plt.figure()

plt.plot(epochs, loss, 'b', label='Training Loss')
plt.title('Training loss')
plt.legend()

plt.show()

In [ ]:
seed_text = "Help me Obi Wan Kenobi, you're my only hope"
next_words = 5
  
for _ in range(next_words):
	token_list = tokenizer.texts_to_sequences([seed_text])[0]
	token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
	predicted = model.predict(token_list, verbose=0)
	predicted = np.argmax(predicted, axis=-1).item()
	output_word = tokenizer.index_word[predicted]
	seed_text += " " + output_word

print(seed_text)